### Libraries

In [ ]:
# to install packages if not installed
# %pip install pandas
# %pip install seaborn
# %pip install statsmodels
# %pip install matplotlib
# %pip install numpy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np

pd.set_option('display.width', 500)

In [ ]:
df = pd.read_excel("data/Data_FE.xlsx", sheet_name="25 Size and BEME portfolios")
df.head()

In [ ]:
# Set rows 1 and 2 as MultiIndex header
df.columns = pd.MultiIndex.from_arrays([df.iloc[0], df.iloc[1]])
df = df.iloc[2:].reset_index(drop=True)

# Convert Date column to Year-Month
date_col = df.columns[0]

df[date_col] = pd.to_datetime(df[date_col].astype(int), format="%Y%m")

df = df.set_index(df.columns[0]) 
df = df.apply(pd.to_numeric)

df.head()

In [ ]:
factors = pd.read_excel("data/Data_FE.xlsx", sheet_name="Fama-French factors")

# Convert Date column to Year-Month
date_col = factors.columns[0] 

factors[date_col] = pd.to_datetime(factors[date_col].astype(int), format="%Y%m")
factors = factors.set_index(factors.columns[0]) 
factors = factors.apply(pd.to_numeric)

factors.head()

In [ ]:
target_df = df.copy()

factors = factors.reindex(target_df.index)

target_df = target_df.sub(factors["RF"], axis=0)
target_df.head()

## Fama-French 3 Factor Model

$$R(t)-RF(t)=a+b[RM(t)-RF(t)]+sSMB(t)+hHML(t)+c(t)$$

### Linear Regression using Statsmodels package

In [ ]:
X = factors[["Mkt-RF", "SMB", "HML"]]
X = sm.add_constant(X)
y = target_df["Small"]["Low"]


model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# X = factors[["Mkt-RF", "SMB", "HML"]]
# X = sm.add_constant(X)

# r_squa = []
# const = []
# Mkt_Rf = []
# SMB = []
# HML = []

# for j in range(len(target_df.columns)):
#     y = target_df.iloc[:, j]
#     model = sm.OLS(y, X).fit()
#     r_squa.append(model.rsquared)
#     parameters = model.params
#     const.append(parameters.iloc[0])
#     Mkt_Rf.append(parameters.iloc[1])
#     SMB.append(parameters.iloc[2])
#     HML.append(parameters.iloc[3])


# dat = {"R-squared": r_squa, "Constant": const, "MKT-Rf" : Mkt_Rf, "SMB" : SMB, "HML" : HML}

# summary_df = pd.DataFrame(dat, index=target_df.columns)
# print(summary_df)


In [ ]:
# Get the multi-index columns from target_df
columns = target_df.columns  # your MultiIndex (Size, BE/ME)

# Store results per year
yearly_results = {}


for year, group in target_df.groupby(target_df.index.year):

    # Get matching factors for this year's dates
    X = factors.loc[group.index, ["Mkt-RF", "SMB", "HML"]]
    X = sm.add_constant(X)

    # empty lists to store the corresponding cofficients
    r_squa, const, Mkt_Rf, SMB, HML = [], [], [], [], []

    for j in range(len(group.columns)):
        y = group.iloc[:, j]
        model = sm.OLS(y, X).fit()

        r_squa.append(model.rsquared)
        const.append(model.params.iloc[0])
        Mkt_Rf.append(model.params.iloc[1])
        SMB.append(model.params.iloc[2])
        HML.append(model.params.iloc[3])

    # Build DataFrame with same structure as target_df columns
    yearly_results[year] = pd.DataFrame(
        {"R-squared": r_squa, "Constant": const, "MKT-Rf": Mkt_Rf, "SMB": SMB, "HML": HML}, index=columns
    )

# combine all years into one DataFrame
summary_df = pd.concat(yearly_results, axis=0)
print(summary_df)


## Macroeconomics Factor Model

In [ ]:
macro_factors = pd.read_excel("data/Data_FE.xlsx", sheet_name="Macroeconomic factors")

# Convert Date column to Year-Month
date_col = macro_factors.columns[0] 

macro_factors[date_col] = pd.to_datetime(macro_factors[date_col].astype(int), format="%Y%m")
macro_factors = macro_factors.set_index(macro_factors.columns[0]) 
macro_factors = macro_factors.apply(pd.to_numeric)

macro_factors.head()

### Linear regression

In [ ]:
X = macro_factors[["Div_growth", "DEF = LT Corp. - LT govt.", "TERM = ST govt.-LT govt."]]
X = sm.add_constant(X)

r_squa = []
const = []
Div_growth = []
DEF = []
TERM = []

for j in range(len(target_df.columns)):
    y = df.iloc[:, j]
    model = sm.OLS(y, X).fit()
    r_squa.append(model.rsquared)
    parameters = model.params
    const.append(parameters.iloc[0])
    Div_growth.append(parameters.iloc[1])
    DEF.append(parameters.iloc[2])
    TERM.append(parameters.iloc[3])


dat = {"R-squared": r_squa, "Constant": const, "Div_growth" : Div_growth, "DEF = LT Corp. - LT govt." : DEF, "TERM = ST govt.-LT govt." : TERM}

summary_macro = pd.DataFrame(dat, index=df.columns)
print(summary_macro)

## In-sample Principle Component Factors

In [ ]:
# excess returns
excess_returns = df.copy()

macro_factors = macro_factors.reindex(excess_returns.index)
excess_returns = excess_returns.sub(macro_factors["TERM = ST govt.-LT govt."], axis=0)
excess_returns.head()

# covariance matrix of excess returns
cova_matrix = excess_returns.cov()

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(cova_matrix)

# eigh returns in ASCENDING order — reverse to get largest first
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]    # 25 × 25, columns are eigenvectors

# extracting 3 largest eigenvectors, the factor weights is 25 x 3
W = eigenvectors[:, :3]

# factor time-series (T x 3)
F = excess_returns.values @ W 

in_sample_factors_df = pd.DataFrame(F, index=excess_returns.index, columns=['In-sample-PC1', 'In-sample-PC2', 'In-sample-PC3'])

in_sample_factors_df.head()

### Linear regression for In-sample PC factors

In [ ]:
X = in_sample_factors_df[["In-sample-PC1", "In-sample-PC2", "In-sample-PC3"]]
X = sm.add_constant(X)

r_squa = []
const = []
In_sample_PC1 = []
In_sample_PC2 = []
In_sample_PC3 = []

for j in range(len(df.columns)):
    y = df.iloc[:, j]
    model = sm.OLS(y, X).fit()
    r_squa.append(model.rsquared)
    parameters = model.params
    const.append(parameters.iloc[0])
    In_sample_PC1.append(parameters.iloc[1])
    In_sample_PC2.append(parameters.iloc[2])
    In_sample_PC3.append(parameters.iloc[3])


dat = {"R-squared": r_squa, "Constant": const, "In-sample-PC1" : In_sample_PC1, "In-sample-PC2" : In_sample_PC2, "In-sample-PC3" : In_sample_PC3}

summary_In_sample = pd.DataFrame(dat, index=df.columns)
print(summary_In_sample)

## Out-sample principle Component Factors

### Even and Odd Monthly Returns dataframes

In [ ]:
# making odd monthly return and even monthly returns dataframe
odd_returns_df = df.copy()
odd_returns_df = odd_returns_df[::1]

even_returns_df = df.copy()
even_returns_df = even_returns_df[::1]

# covariance matrix
cova_matrix_even = even_returns_df.cov()
cova_matrix_odd = odd_returns_df.cov()

In [ ]:
# Eigendecomposition for even
eigenvalues_even, eigenvectors_even = np.linalg.eigh(cova_matrix_even)

# eigh returns in ascending order — reverse to get largest first
idx = np.argsort(eigenvalues_even)[::-1]
eigenvalues_even  = eigenvalues_even[idx]
eigenvectors_even = eigenvectors_even[:, idx]    # 25 × 25, columns are eigenvectors

# extracting 3 largest eigenvectors, the factor weights is 25 x 3
W_even = eigenvectors_even[:, :3]

# Eigendecomposition for odd
eigenvalues_odd, eigenvectors_odd = np.linalg.eigh(cova_matrix_odd)

# eigh returns in ascending order — reverse to get largest first
idx = np.argsort(eigenvalues_odd)[::-1]
eigenvalues_odd  = eigenvalues_odd[idx]
eigenvectors_odd = eigenvectors_odd[:, idx]    # 25 × 25, columns are eigenvectors

# extracting 3 largest eigenvectors, the factor weights is 25 x 3
W_odd = eigenvectors_odd[:, :3]

# factor time-series (T x 25)
F_even = even_returns_df.values @ W_odd
F_odd = odd_returns_df.values @ W_even

# Create empty dataframe with full index
out_sample_factors_df = pd.DataFrame(index=df.index, columns=['Out-sample-PC1', 'Out-sample-PC2', 'Out-sample-PC3'], dtype=float)

# Assign values to correct timestamps
out_sample_factors_df.loc[even_returns_df.index] = F_even
out_sample_factors_df.loc[odd_returns_df.index]  = F_odd

out_sample_factors_df.head()

### Linear regression for Out-sample PC factors

In [ ]:
X = out_sample_factors_df[["Out-sample-PC1", "Out-sample-PC2", "Out-sample-PC3"]]
X = sm.add_constant(X)

r_squa = []
const = []
out_sample_PC1 = []
out_sample_PC2 = []
out_sample_PC3 = []

for j in range(len(df.columns)):
    y = df.iloc[:, j]
    model = sm.OLS(y, X).fit()
    r_squa.append(model.rsquared)
    parameters = model.params
    const.append(parameters.iloc[0])
    out_sample_PC1.append(parameters.iloc[1])
    out_sample_PC2.append(parameters.iloc[2])
    out_sample_PC3.append(parameters.iloc[3])


dat = {"R-squared": r_squa, "Constant": const, "Out-sample-PC1" : out_sample_PC1, "Out-sample-PC2" : out_sample_PC2, "Out-sample-PC3" : out_sample_PC3}

summary_out_sample = pd.DataFrame(dat, index=df.columns)
print(summary_out_sample)